# Teach Jumper to jump *toward* something (Colab, GPU)
Stock `jumper.ref_free_jump` jumps straight up (it even charges a penalty for travelling). This notebook installs a **fork** — `jumper.ref_free_jump_dir` — whose actor also sees a commanded landing offset `(dx, dy)`, and **warm-starts it from Jumper's own trained vertical-jump policy** (`model_5999.pt`), so it only has to learn to *aim*.

**Untested on a GPU** (I had no GPU/mjlab when writing it) — the checkpoint surgery is checked numerically, the environment code is not. If a cell fails, paste the error. A smoke test (cell 7) runs 30 iterations before you commit to hours.

Runtime → Change runtime type → a GPU (A100/L4 ≫ T4). Cells 1–4 are setup, 7 is the smoke test, 8 the long run, 9 renders the result.

In [ ]:
# 1. GPU check
!nvidia-smi -L
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv

In [ ]:
# 2. Jumper pinned (third-party, Apache-2.0) + this repo
import os, sys, subprocess
JUMPER='/content/jumper'; REV='78966f45de6f2b6de3f7cea017f32a39f8b993e6'
SB='/content/spiderbrain-moveset'
if not os.path.exists(JUMPER): !git clone https://github.com/KingKongRobotics/jumper.git {JUMPER}
!cd {JUMPER} && git checkout -q {REV} && git log -1 --format='%h %s'
if not os.path.exists(SB): !git clone https://github.com/adhritpradeepashoka/spiderbrain-moveset.git {SB}
!cd {SB} && git pull -q && git log -1 --format='%h %s'

In [ ]:
# 3. venv with uv (Colab's own venv has no pip) + torch + Jumper
cap = float(subprocess.check_output(['nvidia-smi','--query-gpu=compute_cap','--format=csv,noheader']).decode().split()[0])
CUDA = 'cu128' if cap >= 8.9 else 'cu126'
PY = f'{JUMPER}/.venv/bin/python'
!pip install -q uv
!rm -rf {JUMPER}/.venv
!uv venv --python {sys.executable} {JUMPER}/.venv
!uv pip install --python {PY} torch==2.9.1 torchvision==0.24.1 --index-url https://download.pytorch.org/whl/{CUDA} 2>&1 | tail -3
!cd {JUMPER} && uv pip install --python {PY} -e . 2>&1 | tail -3
!{PY} -c "import warp as wp; wp.init(); print(wp.get_devices())" 

In [ ]:
# 4. install the fork task, build the warm-start checkpoint, check the task is registered
!{PY} {SB}/jumper_fork/install.py {JUMPER}
!{PY} {SB}/jumper_fork/make_warmstart.py {JUMPER}/tasks/jumper/ref_free_jump/out/example/model_5999.pt /content/warm.pt
!cd {JUMPER} && {PY} scripts/train.py --list 2>&1 | grep -i "ref_free"
!cd {JUMPER} && {PY} scripts/train.py --task jumper.ref_free_jump_dir --headless --no-tensorboard --dry-run 2>&1 | tail -3

### 5. (optional) save runs to Google Drive so a disconnect doesn't lose hours
Run this cell and approve; the long run copies checkpoints to `MyDrive/jumper_dir_logs/` every few minutes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/jumper_dir_logs

### 6. What the knobs mean
`JUMP_TARGET_MAX` (default 0.30 m) is the farthest forward target; it ramps in from iteration 100 to 1500 (`JUMP_RAMP_ITERS=100,1500`). 20 % of episodes keep target (0,0) so the vertical jump survives. Lower the max if the first run can't reach it.

In [ ]:
# 7. SMOKE TEST: 30 iterations, 512 envs. Look for: no traceback, 'resumed', reward numbers, landing_err_m / target_fwd_m in the log.
!cd {JUMPER} && JUMP_RAMP_ITERS=0,20 {PY} scripts/train.py --task jumper.ref_free_jump_dir --checkpoint /content/warm.pt --headless --no-tensorboard --num_envs 512 --max-iterations 30 2>&1 | tail -60

In [ ]:
# 8. THE LONG RUN (background; ~3000 iterations). Logs: /content/train.log. Re-run cell 8b to watch it.
import subprocess
env = dict(os.environ, JUMP_TARGET_MAX='0.30', JUMP_RAMP_ITERS='100,1500', MUJOCO_GL='egl')
train = subprocess.Popen(f'cd {JUMPER} && {PY} scripts/train.py --task jumper.ref_free_jump_dir --checkpoint /content/warm.pt --headless --no-tensorboard --max-iterations 3000 > /content/train.log 2>&1', shell=True, env=env)
print('started pid', train.pid)

In [ ]:
# 8b. progress + checkpoint sync to Drive
!grep -E "Learning iteration|landing_err|target_fwd|Mean reward|Traceback|Error" /content/train.log | tail -12
!mkdir -p /content/drive/MyDrive/jumper_dir_logs && cp -ru {JUMPER}/logs /content/drive/MyDrive/jumper_dir_logs/ 2>/dev/null; ls {JUMPER}/logs/*/jumper.ref_free_jump_dir/*/ 2>/dev/null | tail -5

In [ ]:
# 9. RENDER: newest checkpoint, a fixed target (change JUMP_TARGET_X/Y), side camera. Run after training (or on an intermediate checkpoint).
import os
os.environ.update(JUMP_TARGET_X='0.20', JUMP_TARGET_Y='0.0', MUJOCO_GL='egl')
!cd {JUMPER} && {PY} scripts/play.py --task jumper.ref_free_jump_dir --headless --steps 600 --video /content/dir_jump.mp4 --no-video-follow --video-lookat-height 0.08 --video-distance 1.6 --video-azimuth 90 2>&1 | tail -15
from IPython.display import Video
Video('/content/dir_jump.mp4', embed=True, width=640)

In [ ]:
# 10. download the best checkpoint + clip
from google.colab import files
import glob
ck = sorted(glob.glob(f'{JUMPER}/logs/**/jumper.ref_free_jump_dir/**/model_*.pt', recursive=True), key=os.path.getmtime)
print(ck[-3:])
if ck: files.download(ck[-1])
files.download('/content/dir_jump.mp4')